# Metadata

Domain: Mathematics

Subdomain: Real Analysis

References: TODO


# Title

TODO: task title


# Subproblem 1

## Prompt

Describe the subproblem_1 problem here: the scientific setup, what the trainer must implement, and the expected inputs/outputs.

```python
def subproblem_1(...):
    """
    TODO: docstring — inputs, outputs, units.
    """
    ...
```


## Background

TODO: background / derivation / references for subproblem_1.


## Testing Template

In [ ]:
# ============ TEST CASES ============
import numpy as np


def test_case_1():
    """b_0 = 1 for every alpha in (0, 1]."""
    for alpha in [0.1, 0.25, 0.5, 0.75, 0.9, 1.0]:
        b = l1_weights(5, alpha)
        assert np.isclose(b[0], 1.0, atol=1e-14), (
            f"b[0] = {b[0]} for alpha={alpha}, expected 1.0"
        )
    print('test_case_1 passed')


def test_case_2():
    """Weights are strictly positive and strictly decreasing for alpha < 1."""
    b = l1_weights(20, 0.5)
    assert np.all(b > 0), "all weights must be positive"
    assert np.all(np.diff(b) < 0), "weights must be strictly decreasing"
    print('test_case_2 passed')


def test_case_3():
    """alpha = 1 reduces to backward-difference weights: [1, 0, 0, ..., 0].

    Catches an implementation that forgets the special case at alpha=1
    (numpy's 0**0 = 1 makes the direct formula give b[0] = 0, and the
    shifted formula b_j = j^{1-alpha} - (j-1)^{1-alpha} gives b[1] = 0
    but b[0] = nan)."""
    b = l1_weights(5, 1.0)
    expected = np.array([1.0, 0.0, 0.0, 0.0, 0.0, 0.0])
    assert np.allclose(b, expected, atol=1e-14), f"got {b}, expected {expected}"
    print('test_case_3 passed')


def test_case_4():
    """Explicit check at alpha=0.5, j=2: b_2 = sqrt(3) - sqrt(2).

    The off-by-one bug (b_j = j^{1-alpha} - (j-1)^{1-alpha}) gives
    b_2 = sqrt(2) - 1 ~ 0.414 instead of sqrt(3) - sqrt(2) ~ 0.318.
    Both agree at j=0 and j=1, so only j >= 2 catches the bug."""
    b = l1_weights(5, 0.5)
    expected_b2 = np.sqrt(3.0) - np.sqrt(2.0)
    assert np.isclose(b[2], expected_b2, atol=1e-14), (
        f"b[2] = {b[2]}, expected {expected_b2} (off-by-one bug?)"
    )
    print('test_case_4 passed')


def test_case_5():
    """Explicit check at alpha=0.25, j=1 and j=3, against the closed form."""
    alpha = 0.25
    b = l1_weights(5, alpha)
    for j in [1, 3]:
        expected = (j + 1) ** (1 - alpha) - j ** (1 - alpha)
        assert np.isclose(b[j], expected, atol=1e-14), (
            f"b[{j}] = {b[j]}, expected {expected}"
        )
    print('test_case_5 passed')


def test_case_6():
    """Input validation: alpha outside (0, 1] or n < 0 must raise ValueError."""
    for bad_alpha in [0.0, -0.5, 1.5, 2.0]:
        try:
            l1_weights(5, bad_alpha)
            assert False, f"expected ValueError for alpha={bad_alpha}"
        except ValueError:
            pass
    for bad_n in [-1, -10]:
        try:
            l1_weights(bad_n, 0.5)
            assert False, f"expected ValueError for n={bad_n}"
        except ValueError:
            pass
    print('test_case_6 passed')


## Solution

In [ ]:
import numpy as np


def l1_weights(n, alpha):
    """L1 convolution weights b_j = (j+1)^{1-alpha} - j^{1-alpha}, j=0..n."""
    if not (0.0 < alpha <= 1.0):
        raise ValueError("alpha must be in (0, 1]")
    if not isinstance(n, (int, np.integer)) or isinstance(n, bool) or n < 0:
        raise ValueError("n must be a non-negative integer")

    n = int(n)
    b = np.zeros(n + 1, dtype=float)
    b[0] = 1.0
    if n >= 1:
        j = np.arange(1, n + 1, dtype=float)
        b[1:] = (j + 1.0) ** (1.0 - alpha) - j ** (1.0 - alpha)
    return b

# Subproblem 2

## Prompt

Describe the subproblem_2 problem here: the scientific setup, what the trainer must implement, and the expected inputs/outputs.

```python
def subproblem_2(...):
    """
    TODO: docstring — inputs, outputs, units.
    """
    ...
```


## Background

TODO: background / derivation / references for subproblem_2.


## Testing Template

In [ ]:
# ============ TEST CASES ============
import numpy as np
import math


def test_case_1():
    """D[0] = 0 and the output has the same shape as the grid."""
    f = lambda t: t ** 2
    t = np.linspace(0.0, 1.0, 11)
    D = caputo_l1(f, t, 0.5)
    assert D.shape == t.shape, f"expected shape {t.shape}, got {D.shape}"
    assert D[0] == 0.0, f"D[0] = {D[0]}, expected 0"
    print('test_case_1 passed')


def test_case_2():
    """Constant f: D^alpha (const) = 0 exactly."""
    f = lambda t: 3.7
    t = np.linspace(0.0, 1.0, 21)
    for alpha in [0.25, 0.5, 0.75, 1.0]:
        D = caputo_l1(f, t, alpha)
        assert np.allclose(D, 0.0, atol=1e-13), (
            f"constant f should give D = 0 for alpha={alpha}, got {D}"
        )
    print('test_case_2 passed')


def test_case_3():
    """Linear f: L1 is EXACT for f(t) = t.
    D^alpha t = t^{1-alpha} / Gamma(2-alpha) exactly. Very tight tolerance."""
    f = lambda t: t
    for alpha in [0.25, 0.5, 0.75, 1.0]:
        t = np.linspace(0.0, 2.0, 21)
        D = caputo_l1(f, t, alpha)
        expected = np.zeros_like(t)
        expected[1:] = t[1:] ** (1.0 - alpha) / math.gamma(2.0 - alpha)
        assert np.allclose(D, expected, atol=1e-12, rtol=1e-12), (
            f"L1 exactness on f=t failed for alpha={alpha}: "
            f"max |err| = {np.max(np.abs(D - expected)):.3e}"
        )
    print('test_case_3 passed')


def test_case_4():
    """alpha = 1 reduces to the backward difference (f(t_k)-f(t_{k-1}))/tau."""
    f = lambda t: np.sin(t)
    t = np.linspace(0.0, 1.0, 31)
    tau = t[1] - t[0]
    D = caputo_l1(f, t, 1.0)
    expected = np.zeros_like(t)
    expected[1:] = (f(t[1:]) - f(t[:-1])) / tau
    assert np.allclose(D, expected, atol=1e-14, rtol=1e-14), (
        f"alpha=1 should equal backward difference, max err = "
        f"{np.max(np.abs(D - expected)):.3e}"
    )
    print('test_case_4 passed')


def test_case_5():
    """Convergence order: for f(t) = t^2 at alpha=0.5, refining the grid
    should reduce the error roughly as tau^{2-alpha} = tau^{1.5}.

    Ratio of successive errors should be near 2^{1.5} ~ 2.83."""
    alpha = 0.5
    true_coeff = math.gamma(3.0) / math.gamma(3.0 - alpha)   # D^alpha t^2 coeff

    def max_err(N):
        t = np.linspace(0.0, 1.0, N + 1)
        f = lambda tt: tt ** 2
        D = caputo_l1(f, t, alpha)
        expected = np.zeros_like(t)
        expected[1:] = true_coeff * t[1:] ** (2.0 - alpha)
        return np.max(np.abs(D[1:] - expected[1:]))

    e1 = max_err(40)
    e2 = max_err(80)
    ratio = e1 / e2
    assert 2.0 < ratio < 4.5, (
        f"expected error ratio near 2^{{1.5}} ~ 2.83, got {ratio:.2f} "
        f"(e1 = {e1:.3e}, e2 = {e2:.3e})"
    )
    print('test_case_5 passed')


def test_case_6():
    """Input validation: non-uniform grid, too-short grid, bad alpha."""
    f = lambda t: t
    # Non-uniform grid
    t_bad = np.array([0.0, 0.1, 0.25, 0.4, 0.6])
    try:
        caputo_l1(f, t_bad, 0.5)
        assert False, "expected ValueError for non-uniform grid"
    except ValueError:
        pass
    # Too short (only one point)
    try:
        caputo_l1(f, np.array([0.0]), 0.5)
        assert False, "expected ValueError for grid with fewer than 2 points"
    except ValueError:
        pass
    # Bad alpha
    t = np.linspace(0.0, 1.0, 11)
    for bad in [0.0, -0.5, 1.5]:
        try:
            caputo_l1(f, t, bad)
            assert False, f"expected ValueError for alpha={bad}"
        except ValueError:
            pass
    print('test_case_6 passed')

## Solution

In [ ]:
import numpy as np
import math


def l1_weights(n, alpha):
    """L1 convolution weights (inlined for isolation)."""
    if not (0.0 < alpha <= 1.0):
        raise ValueError("alpha must be in (0, 1]")
    if not isinstance(n, (int, np.integer)) or isinstance(n, bool) or n < 0:
        raise ValueError("n must be a non-negative integer")
    n = int(n)
    b = np.zeros(n + 1, dtype=float)
    b[0] = 1.0
    if n >= 1:
        j = np.arange(1, n + 1, dtype=float)
        b[1:] = (j + 1.0) ** (1.0 - alpha) - j ** (1.0 - alpha)
    return b


def caputo_l1(f, t, alpha):
    """L1 approximation of the Caputo derivative D^alpha f on a uniform grid."""
    if not (0.0 < alpha <= 1.0):
        raise ValueError("alpha must be in (0, 1]")

    t = np.asarray(t, dtype=float)
    if t.ndim != 1:
        raise ValueError("t must be a 1-D array")
    if t.size < 2:
        raise ValueError("grid must have at least 2 points")

    tau = t[1] - t[0]
    if not np.allclose(np.diff(t), tau, rtol=1e-9, atol=1e-12):
        raise ValueError("grid must be uniform")

    N = t.size - 1
    f_vals = np.array([float(f(ti)) for ti in t], dtype=float)
    b = l1_weights(N, alpha)
    normalizer = math.gamma(2.0 - alpha) * tau ** alpha

    D = np.zeros(N + 1, dtype=float)
    for n in range(1, N + 1):
        acc = 0.0
        for k in range(1, n + 1):
            acc += b[n - k] * (f_vals[k] - f_vals[k - 1])
        D[n] = acc / normalizer
    return D

# Main Problem

## Prompt

Describe the main_problem problem here: the scientific setup, what the trainer must implement, and the expected inputs/outputs.

```python
def main_problem(...):
    """
    TODO: docstring — inputs, outputs, units.
    """
    ...
```


## Background

TODO: background / derivation / references for main_problem.


## Testing Template

In [ ]:
# ============ TEST CASES ============
import numpy as np
import math


def test_case_1():
    """alpha=1: exact discrete backward-Euler solution u_n = u0/(1+lambda*tau)^n."""
    alpha, lam, T, N, u0 = 1.0, 2.0, 1.0, 100, 1.0
    t, u = fractional_relaxation_solve(alpha, lam, T, N, u0)
    tau = T / N
    expected = u0 / (1.0 + lam * tau) ** np.arange(N + 1)
    assert np.allclose(u, expected, rtol=1e-13, atol=1e-13), (
        f"alpha=1 backward-Euler regression failed: "
        f"max |err| = {np.max(np.abs(u - expected)):.3e}"
    )
    print('test_case_1 passed')


def test_case_2():
    """Grid layout: t has N+1 uniformly spaced points, t[0]=0, t[N]=T, u[0]=u0."""
    alpha, lam, T, N, u0 = 0.5, 1.0, 2.0, 40, 3.7
    t, u = fractional_relaxation_solve(alpha, lam, T, N, u0)
    assert t.shape == (N + 1,)
    assert u.shape == (N + 1,)
    assert np.isclose(t[0], 0.0, atol=1e-15)
    assert np.isclose(t[N], T, atol=1e-15)
    assert np.isclose(u[0], u0, atol=1e-15)
    tau = T / N
    assert np.allclose(np.diff(t), tau, atol=1e-14)
    print('test_case_2 passed')


def test_case_3():
    """alpha=0.5, small-t comparison to the Mittag-Leffler closed form
    E_{1/2}(z) = exp(z^2) erfc(-z), i.e.
    u(t) = u0 * exp(lambda^2 * t) * erfc(lambda * sqrt(t)).

    Tolerance accounts for the L1 discretization error O(tau^{2-alpha})."""
    alpha, lam, T, N, u0 = 0.5, 1.0, 0.5, 200, 1.0
    t, u = fractional_relaxation_solve(alpha, lam, T, N, u0)
    exact = u0 * np.exp(lam ** 2 * t) * np.array([math.erfc(lam * math.sqrt(ti)) for ti in t])
    err = np.max(np.abs(u - exact))
    assert err < 5e-3, (
        f"alpha=0.5 Mittag-Leffler comparison failed: max |err| = {err:.3e}"
    )
    print('test_case_3 passed')


def test_case_4():
    """Qualitative: u[0]=u0, u is strictly decreasing, 0 < u[n] <= u0."""
    alpha, lam, T, N, u0 = 0.5, 2.0, 3.0, 60, 1.0
    t, u = fractional_relaxation_solve(alpha, lam, T, N, u0)
    assert np.isclose(u[0], u0, atol=1e-15)
    assert np.all(np.diff(u) < 0), "solution must be strictly decreasing"
    assert np.all(u > 0.0), "solution must stay positive"
    assert np.all(u <= u0 + 1e-15), "solution must not exceed the initial value"
    print('test_case_4 passed')


def test_case_5():
    """Stability with large lambda * tau^alpha: no blow-up, monotone decay.

    An explicit-Euler-style treatment would oscillate or grow for these
    parameters. The implicit L1 scheme must remain bounded and monotone."""
    alpha, lam, T, N, u0 = 0.5, 100.0, 0.5, 20, 1.0
    t, u = fractional_relaxation_solve(alpha, lam, T, N, u0)
    assert np.all(np.isfinite(u)), "solution must be finite"
    assert np.all(u > 0.0) and np.all(u <= u0 + 1e-15), (
        "solution must remain in [0, u0]; an explicit scheme would leave this bound"
    )
    assert np.all(np.diff(u) < 0), "solution must decay monotonically"
    print('test_case_5 passed')


def test_case_6():
    """Input validation: alpha outside (0, 1], lam <= 0, T <= 0, N < 1."""
    for bad in [0.0, -0.5, 1.5]:
        try:
            fractional_relaxation_solve(bad, 1.0, 1.0, 10)
            assert False, f"expected ValueError for alpha={bad}"
        except ValueError:
            pass
    for bad in [0.0, -1.0]:
        try:
            fractional_relaxation_solve(0.5, bad, 1.0, 10)
            assert False, f"expected ValueError for lam={bad}"
        except ValueError:
            pass
    for bad in [0.0, -1.0]:
        try:
            fractional_relaxation_solve(0.5, 1.0, bad, 10)
            assert False, f"expected ValueError for T={bad}"
        except ValueError:
            pass
    for bad in [0, -1]:
        try:
            fractional_relaxation_solve(0.5, 1.0, 1.0, bad)
            assert False, f"expected ValueError for N={bad}"
        except ValueError:
            pass
    print('test_case_6 passed')


def test_case_7():
    """Monotone refinement: on the same physical problem, a finer grid
    must produce a solution that is closer to (not farther from) the
    alpha=0.5 closed form."""
    alpha, lam, T, u0 = 0.5, 1.0, 0.5, 1.0

    def err_vs_closed_form(N):
        t, u = fractional_relaxation_solve(alpha, lam, T, N, u0)
        exact = u0 * np.exp(lam ** 2 * t) * np.array(
            [math.erfc(lam * math.sqrt(ti)) for ti in t]
        )
        return np.max(np.abs(u - exact))

    e_coarse = err_vs_closed_form(50)
    e_fine = err_vs_closed_form(400)
    assert e_fine < e_coarse, (
        f"refining the grid should reduce the error; "
        f"coarse={e_coarse:.3e}, fine={e_fine:.3e}"
    )
    print('test_case_7 passed')

## Solution

In [ ]:
import numpy as np
import math


def l1_weights(n, alpha):
    """L1 convolution weights (inlined for isolation)."""
    if not (0.0 < alpha <= 1.0):
        raise ValueError("alpha must be in (0, 1]")
    if not isinstance(n, (int, np.integer)) or isinstance(n, bool) or n < 0:
        raise ValueError("n must be a non-negative integer")
    n = int(n)
    b = np.zeros(n + 1, dtype=float)
    b[0] = 1.0
    if n >= 1:
        j = np.arange(1, n + 1, dtype=float)
        b[1:] = (j + 1.0) ** (1.0 - alpha) - j ** (1.0 - alpha)
    return b


def fractional_relaxation_solve(alpha, lam, T, N, u0=1.0):
    """Solve D^alpha u = -lam * u with u(0) = u0, via the implicit L1 scheme."""
    if not (0.0 < alpha <= 1.0):
        raise ValueError("alpha must be in (0, 1]")
    if lam <= 0.0:
        raise ValueError("lam must be positive")
    if T <= 0.0:
        raise ValueError("T must be positive")
    if not isinstance(N, (int, np.integer)) or isinstance(N, bool) or N < 1:
        raise ValueError("N must be an integer >= 1")

    N = int(N)
    tau = T / N
    t = np.linspace(0.0, T, N + 1)
    b = l1_weights(N, alpha)
    G = math.gamma(2.0 - alpha) * (tau ** alpha) * lam
    denom = 1.0 + G

    u = np.zeros(N + 1, dtype=float)
    u[0] = float(u0)
    for n in range(1, N + 1):
        history = 0.0
        for k in range(1, n):
            history += b[n - k] * (u[k] - u[k - 1])
        u[n] = (u[n - 1] - history) / denom
    return t, u